## This is all the import


In [3]:
# verify_setup.py
import torch
import transformers
from torch.cuda import get_device_properties

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"CuDNN Version: {torch.backends.cudnn.version()}")
print(f"GPU Count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = get_device_properties(i)
    print(f"\nGPU {i}: {props.name}")
    print(f"  Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute Capability: {props.major}.{props.minor}")
    print(f"  Max Threads per Block: {props}")

# Check VRAM usage
print(f"\nCurrent VRAM Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Max VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch Version: 2.9.1+cu130
CUDA Available: True
CUDA Version: 13.0
CuDNN Version: 91300
GPU Count: 1

GPU 0: NVIDIA GeForce RTX 5070 Ti
  Memory: 16.6 GB
  Compute Capability: 12.0
  Max Threads per Block: _CudaDeviceProperties(name='NVIDIA GeForce RTX 5070 Ti', major=12, minor=0, total_memory=15817MB, multi_processor_count=70, uuid=3444b2e3-6da2-25db-fdfe-409398752fc9, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=48MB)

Current VRAM Used: 0.00 GB
Max VRAM Available: 16.6 GB


In [1]:
"""
Discrete Diffusion Language Model Architecture
Based on: https://arxiv.org/abs/2211.15029
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple, List
from dataclasses import dataclass



@dataclass
class DiffusionLLMConfig:
    """Configuration for Diffusion-based LLM"""
    
    # Model architecture
    vocab_size: int = 32000
    hidden_size: int = 576          # ← Reduced for 8GB
    num_hidden_layers: int = 12
    num_attention_heads: int = 9
    num_key_value_heads: int = 3
    intermediate_size: int = 1536
    max_seq_length: int = 512
    
    # Diffusion parameters
    num_diffusion_steps: int = 50   # ← Key: number of denoising steps
    diffusion_schedule: str = "cosine"  # linear, sqrt, cosine
    noise_pred_type: str = "sample"     # sample, mean, logits
    
    # Training
    timestep_embedding_dim: int = 128
    rms_norm_eps: float = 1e-5
    
    def __post_init__(self):
        assert self.hidden_size % self.num_attention_heads == 0
        assert self.hidden_size % self.num_key_value_heads == 0


class NoiseSchedule:
    """Handles noise schedule for diffusion process."""
    
    def __init__(self, num_steps: int = 50, schedule_type: str = "cosine"):
        self.num_steps = num_steps
        self.schedule_type = schedule_type
        
        # Pre-compute schedules
        if schedule_type == "cosine":
            self.alphas = self._cosine_schedule()
        elif schedule_type == "linear":
            self.alphas = torch.linspace(1.0, 0.0, num_steps)
        elif schedule_type == "sqrt":
            self.alphas = torch.sqrt(torch.linspace(1.0, 0.0, num_steps))
        
        # Precompute cumulative products
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat(
            [torch.ones(1), self.alphas_cumprod[:-1]]
        )
        
        # Precompute betas (noise levels)
        self.betas = 1.0 - self.alphas
        self.sqrt_betas = torch.sqrt(self.betas)
        self.sqrt_alphas = torch.sqrt(self.alphas)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
    
    def _cosine_schedule(self) -> torch.Tensor:
        """Cosine annealing schedule (better for language)."""
        steps = torch.linspace(0, 1, self.num_steps + 1)
        alphas = torch.cos(((steps + 0.008) / 1.008) * math.pi * 0.5) ** 2
        alphas = alphas / alphas[0]
        return alphas[:-1]
    
    def get_noise_level(self, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get noise level for timestep t.
        
        Args:
            t: timestep indices [batch_size]
        
        Returns:
            sqrt_alpha_t: scaling for signal
            sqrt_one_minus_alpha_t: scaling for noise
        """
        sqrt_alpha_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1)
        sqrt_one_minus_alpha_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1)
        
        return sqrt_alpha_t, sqrt_one_minus_alpha_t


class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding."""
    
    def __init__(self, embedding_dim: int):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Positional encoding
        half_dim = embedding_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim) * -emb)
        self.register_buffer("emb", emb)
    
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: [batch_size] timestep indices
        
        Returns:
            [batch_size, embedding_dim] timestep embeddings
        """
        emb = t[:, None] * self.emb[None, :]  # [batch, half_dim]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb

class DiffusionAttention(nn.Module):
    """Attention layer with timestep conditioning."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        
        # Projections
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Standard causal self-attention."""
        batch_size, seq_len, _ = hidden_states.shape
        
        # Project and reshape
        q = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_attention_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        
        # Repeat for GQA
        num_rep = self.num_attention_heads // self.num_key_value_heads
        k = k.unsqueeze(2).repeat(1, 1, num_rep, 1, 1).reshape(batch_size, self.num_attention_heads, seq_len, self.head_dim)
        v = v.unsqueeze(2).repeat(1, 1, num_rep, 1, 1).reshape(batch_size, self.num_attention_heads, seq_len, self.head_dim)
        
        # Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal mask
        if attention_mask is None:
            causal_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
            scores = scores + causal_mask.to(scores.device)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        # Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.o_proj(attn_output)

class DiffusionBlock(nn.Module):
    """Transformer block with diffusion conditioning."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        
        # Pre-norm
        self.input_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attn_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        
        # Attention
        self.attention = DiffusionAttention(config)
        
        # FFN
        self.mlp = nn.Sequential(
            nn.Linear(config.hidden_size, config.intermediate_size),
            nn.SiLU(),
            nn.Linear(config.intermediate_size, config.hidden_size),
        )
        
        # Timestep conditioning (adaptive instance normalization)
        self.time_mlp = nn.Sequential(
            nn.Linear(config.timestep_embedding_dim, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size * 2),
        )
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        time_emb: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            hidden_states: [batch, seq_len, hidden_size]
            time_emb: [batch, timestep_embedding_dim]
            attention_mask: optional causal mask
        
        Returns:
            [batch, seq_len, hidden_size]
        """
        # Get time conditioning (AdaIN style)
        time_cond = self.time_mlp(time_emb)  # [batch, hidden_size * 2]
        time_scale, time_shift = time_cond.chunk(2, dim=-1)  # Each [batch, hidden_size]
        
        # Self-attention with residual
        residual = hidden_states
        hidden_states = self.input_norm(hidden_states)
        hidden_states = self.attention(hidden_states, attention_mask)
        hidden_states = hidden_states + residual
        
        # FFN with residual and time conditioning
        residual = hidden_states
        hidden_states = self.post_attn_norm(hidden_states)
        
        # Apply time conditioning (scale and shift)
        hidden_states = hidden_states * (1.0 + time_scale.unsqueeze(1)) + time_shift.unsqueeze(1)
        
        # FFN
        hidden_states = self.mlp(hidden_states)
        hidden_states = hidden_states + residual
        
        return hidden_states

class DiffusionLanguageModel(nn.Module):
    """Diffusion-based Language Model."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.config = config
        
        # Embeddings
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        
        # Timestep embedding
        self.timestep_embed = TimestepEmbedding(config.timestep_embedding_dim)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            DiffusionBlock(config) for _ in range(config.num_hidden_layers)
        ])
        
        # Output normalization and projection
        self.final_norm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.logits_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        # Tie embeddings and output
        self.logits_head.weight = self.embed_tokens.weight
        
        # Noise schedule
        self.noise_schedule = NoiseSchedule(config.num_diffusion_steps, config.diffusion_schedule)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(
        self,
        x_t: torch.Tensor,
        t: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass for diffusion model.
        
        Args:
            x_t: [batch, seq_len] noisy token embeddings (already embedded)
            t: [batch] timestep indices
            attention_mask: optional causal mask
        
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        batch_size = x_t.shape[0]
        
        # Embed timestep
        time_emb = self.timestep_embed(t)  # [batch, timestep_embedding_dim]
        
        # Pass through transformer
        hidden_states = x_t
        for block in self.transformer_blocks:
            hidden_states = block(hidden_states, time_emb, attention_mask)
        
        # Final output
        hidden_states = self.final_norm(hidden_states)
        logits = self.logits_head(hidden_states)
        
        return logits

class DiffusionLLMForTraining(nn.Module):
    """Wrapper for training Diffusion LLM."""
    
    def __init__(self, config: DiffusionLLMConfig):
        super().__init__()
        self.config = config
        self.model = DiffusionLanguageModel(config)
        self.noise_schedule = self.model.noise_schedule
        
        # Embeddings (separate to allow token embedding)
        self.embed_tokens = self.model.embed_tokens
    
    def forward(
        self,
        input_ids: torch.Tensor,
        t: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Training forward pass.
        
        Args:
            input_ids: [batch, seq_len] clean token IDs
            t: [batch] timesteps (random if None)
        
        Returns:
            logits: [batch, seq_len, vocab_size] model predictions
            loss: scalar loss
        """
        batch_size, seq_len = input_ids.shape
        
        # Sample timesteps if not provided
        if t is None:
            t = torch.randint(0, self.config.num_diffusion_steps, (batch_size,)).to(input_ids.device)
        
        # Embed clean tokens
        x_0 = self.embed_tokens(input_ids)  # [batch, seq_len, hidden_size]
        
        # Sample noise
        noise = torch.randn_like(x_0)
        
        # Add noise (forward diffusion process)
        sqrt_alpha_t, sqrt_one_minus_alpha_t = self.noise_schedule.get_noise_level(t)
        sqrt_alpha_t = sqrt_alpha_t.to(x_0.device)
        sqrt_one_minus_alpha_t = sqrt_one_minus_alpha_t.to(x_0.device)
        
        x_t = sqrt_alpha_t * x_0 + sqrt_one_minus_alpha_t * noise
        
        # Forward through model (predict noise or mean)
        logits = self.model(x_t, t)
        
        # Predict noise from logits (convert to continuous embedding space)
        noise_pred = logits
        
        # MSE loss on noise prediction (standard diffusion loss)
        # In practice, we'd use cross-entropy on discrete tokens, but for now simple MSE
        loss = F.mse_loss(noise_pred, noise.unsqueeze(-1).expand_as(noise_pred))
        
        return logits, loss
    
    @torch.no_grad()
    def generate(
        self,
        batch_size: int = 1,
        seq_len: int = 256,
        num_inference_steps: int = 50,
    ) -> torch.Tensor:
        """
        Generate text using reverse diffusion process.
        
        Args:
            batch_size: number of sequences to generate
            seq_len: sequence length
            num_inference_steps: number of reverse steps
        
        Returns:
            [batch_size, seq_len] generated token IDs
        """
        device = self.model.embed_tokens.weight.device
        
        # Start from pure noise
        x_t = torch.randn(batch_size, seq_len, self.config.hidden_size).to(device)
        
        # Reverse diffusion process
        for step in range(self.config.num_diffusion_steps - 1, -1, -1):
            t = torch.full((batch_size,), step, dtype=torch.long).to(device)
            
            # Predict noise
            with torch.no_grad():
                noise_pred = self.model(x_t, t)
            
            # Denoise
            alpha_t = self.noise_schedule.alphas[step]
            alpha_t_prev = self.noise_schedule.alphas_cumprod_prev[step]
            
            # Simplified denoising (in practice would use proper reverse formula)
            x_t = (x_t - (1 - alpha_t) * noise_pred) / (alpha_t ** 0.5)
            
            # Add noise for next step (except last)
            if step > 0:
                noise = torch.randn_like(x_t)
                x_t = x_t + (1 - alpha_t_prev) ** 0.5 * noise
        
        # Decode to tokens
        # In practice, would use nearest neighbor or other decoding
        x_0 = x_t
        # Simple nearest neighbor in embedding space
        distances = torch.cdist(x_0.reshape(-1, self.config.hidden_size), 
                               self.embed_tokens.weight)  # [batch*seq, vocab]
        token_ids = torch.argmin(distances, dim=1)
        token_ids = token_ids.reshape(batch_size, seq_len)
        
        return token_ids

# training the LLM

In [8]:
# train_diffusion_llm_8gb.py
"""
Training script for Diffusion-based LLM on 8GB VRAM
Smaller model: 300M parameters with diffusion
"""

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup, AutoTokenizer
from datasets import load_from_disk
from tqdm import tqdm
import os
import json


# # Import from architecture file
# from diffusion_llm_architecture import DiffusionLLMConfig, DiffusionLLMForTraining


class DiffusionTrainingConfig:
    """Training configuration for 8GB GPU."""
    
    # Model
    vocab_size = 32000
    hidden_size = 576        # Smaller than standard
    num_hidden_layers = 12
    num_attention_heads = 9
    num_key_value_heads = 3
    intermediate_size = 1536
    max_seq_length = 512
    
    # Diffusion
    num_diffusion_steps = 50
    diffusion_schedule = "cosine"
    timestep_embedding_dim = 128
    
    # Training
    num_epochs = 3
    batch_size = 2
    gradient_accumulation_steps = 4
    learning_rate = 5e-4
    warmup_steps = 200
    max_grad_norm = 1.0
    weight_decay = 0.01
    
    # Optimization
    use_gradient_checkpointing = True
    use_flash_attention = False  # Not critical for diffusion
    mixed_precision = "fp16"
    use_8bit_optimizer = True
    
    # Logging
    logging_steps = 50
    save_steps = 200
    eval_steps = 200
    
    # Data
    dataset_path = "./data/tokenized"
    output_dir = "./diffusion_checkpoints"
    
    # Hardware
    device = "cuda"
    num_workers = 2


def setup_model_and_tokenizer(config: DiffusionTrainingConfig):
    """Initialize model and tokenizer."""
    
    # Create config
    model_config = DiffusionLLMConfig(
        vocab_size=config.vocab_size,
        hidden_size=config.hidden_size,
        num_hidden_layers=config.num_hidden_layers,
        num_attention_heads=config.num_attention_heads,
        num_key_value_heads=config.num_key_value_heads,
        intermediate_size=config.intermediate_size,
        max_seq_length=config.max_seq_length,
        num_diffusion_steps=config.num_diffusion_steps,
        diffusion_schedule=config.diffusion_schedule,
        timestep_embedding_dim=config.timestep_embedding_dim,
    )
    
    # Create model
    model = DiffusionLLMForTraining(model_config)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {total_params / 1e9:.3f}B")
    
    # Load tokenizer (for reference)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer, model_config


def train_diffusion_lm():
    """Main training loop."""
    
    config = DiffusionTrainingConfig()
    os.makedirs(config.output_dir, exist_ok=True)
    
    print("[1/5] Setting up model and tokenizer...")
    model, tokenizer, model_config = setup_model_and_tokenizer(config)
    
    # Move to GPU
    model = model.to(config.device)
    
    print("[2/5] Loading datasets...")
    train_dataset = load_from_disk(f"{config.dataset_path}/train")
    eval_dataset = load_from_disk(f"{config.dataset_path}/test")
    
    # Subset for 8GB training
    train_dataset = train_dataset.select(range(min(5000, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(1000, len(eval_dataset))))
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Evaluation samples: {len(eval_dataset)}")
    
    print("[3/5] Creating dataloaders...")
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=False,
        drop_last=True,
    )
    
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=False,
    )
    
    print("[4/5] Setting up optimizer and scheduler...")
    
    # Use 8-bit optimizer to save memory
    from bitsandbytes.optim import AdamW8bit
    
    optimizer = AdamW8bit(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95),
    )
    
    total_steps = len(train_loader) * config.num_epochs // config.gradient_accumulation_steps
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=total_steps,
    )
    
    # Mixed precision scaler
    scaler = GradScaler()
    
    # Save config
    with open(f"{config.output_dir}/training_config.json", "w") as f:
        json.dump({
            "model_config": model_config.__dict__,
            "training_config": config.__dict__,
        }, f, indent=2, default=str)
    
    print("\n" + "="*70)
    print("DIFFUSION LLM TRAINING STARTED (8GB VRAM)")
    print("="*70)
    # print(f"Model: {total_params / 1e9:.3f}B parameters")
    print(f"Dataset: {len(train_dataset)} training samples")
    print(f"Batch size: {config.batch_size}")
    print(f"Gradient accumulation: {config.gradient_accumulation_steps}")
    print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
    print(f"Diffusion steps: {config.num_diffusion_steps}")
    print(f"Total training steps: {total_steps}")
    print("="*70 + "\n")
    
    # Training loop
    for epoch in range(config.num_epochs):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")
        
        for step, batch in enumerate(pbar):
            # Prepare batch
            print(f"Preparing batch...{torch.tensor(batch["input_ids"])}")
            input_ids = batch["input_ids"].to(config.device)
            
            # Forward pass with mixed precision
            with autocast(dtype=torch.float16):
                logits, loss = model(input_ids)
                loss = loss / config.gradient_accumulation_steps
            
            # Backward pass
            scaler.scale(loss).backward()
            total_loss += loss.item()
            
            # Gradient accumulation step
            if (step + 1) % config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            
            # Logging
            if (step + 1) % config.logging_steps == 0:
                avg_loss = total_loss / config.logging_steps
                vram_used = torch.cuda.memory_allocated() / 1e9
                pbar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "vram": f"{vram_used:.2f}GB",
                    "lr": f"{scheduler.get_last_lr()[0]:.2e}"
                })
                total_loss = 0
            
            # Save checkpoint
            if (step + 1) % config.save_steps == 0:
                checkpoint_dir = f"{config.output_dir}/checkpoint-{epoch}-{step}"
                os.makedirs(checkpoint_dir, exist_ok=True)
                torch.save(model.state_dict(), f"{checkpoint_dir}/model.pt")
                print(f"\nSaved checkpoint to {checkpoint_dir}")
        
        # Evaluation
        print("\nEvaluating...")
        model.eval()
        eval_loss = 0
        
        with torch.no_grad():
            for batch in tqdm(eval_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(config.device)
                _, loss = model(input_ids)
                eval_loss += loss.item()
        
        avg_eval_loss = eval_loss / len(eval_loader)
        print(f"Epoch {epoch+1} - Eval Loss: {avg_eval_loss:.4f}\n")
    
    # Final save
    final_dir = f"{config.output_dir}/final"
    os.makedirs(final_dir, exist_ok=True)
    torch.save(model.state_dict(), f"{final_dir}/model.pt")
    print(f"\nTraining complete! Model saved to {final_dir}")


if __name__ == "__main__":
    train_diffusion_lm()

[1/5] Setting up model and tokenizer...
Model Parameters: 0.059B


/tmp/ipykernel_25296/2325095113.py:162: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[2/5] Loading datasets...
Training samples: 5000
Evaluation samples: 1000
[3/5] Creating dataloaders...
[4/5] Setting up optimizer and scheduler...

DIFFUSION LLM TRAINING STARTED (8GB VRAM)
Dataset: 5000 training samples
Batch size: 2
Gradient accumulation: 4
Effective batch size: 8
Diffusion steps: 50
Total training steps: 1875



Epoch 1/3:   0%|          | 0/2500 [00:02<?, ?it/s]


TypeError: only integer tensors of a single element can be converted to an index